# Liu2024 Source `.mat` + MOABB-style S-JEPA preprocessing + CSP/FBCSP

Classical CSP/FBCSP sanity-baseline notebook for Liu2024 source `.mat` files.

The preprocessing path is intentionally matched to the MOABB S-JEPA notebook as closely as the source-trial format allows:

```text
select Liu EEG channels → convert source µV to MNE volts → average reference → resample → bandpass → scale back to µV → crop fixed S-JEPA window
```

Liu-specific code still handles the `.mat` loader, CPz/EOG/marker removal, and fixed MI-window extraction.


In [ ]:
import os
import re
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat
from scipy.signal import butter, sosfiltfilt

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

try:
    import mne
    from mne.decoding import CSP
except Exception as exc:
    raise RuntimeError('This notebook needs MNE installed: pip install mne') from exc

print('Runtime Environment:')
print(f'  Python: {os.sys.version}')
print(f'  MNE:    {mne.__version__}')
print(f'  Workdir: {Path.cwd()}')

## 1. Paths and run settings

In [ ]:
WORKING_DIR = Path.cwd().parent.parent
SOURCE_EXTRACT_DIR = WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"
ARTIFACT_DIR = WORKING_DIR / 'artifacts' / 'liu2024-source-mat-sjepa-preprocess-csp-fbcsp' / datetime.now().strftime('%Y%m%d_%H%M%S')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = ARTIFACT_DIR / 'run.log'

# S-JEPA-style preprocessing constants.
SOURCE_SFREQ = 500
SFREQ = 128
BANDPASS_LOW = 0.5
BANDPASS_HIGH = 40.0

# Pierre / S-JEPA-compatible downstream length.
TARGET_WINDOW_SAMPLES = 537  # kept unchanged to preserve the existing S-JEPA windowing setup
TARGET_WINDOW_DURATION_S = TARGET_WINDOW_SAMPLES / SFREQ

# Liu source trial timing: 8 seconds total at 500 Hz.
# For S-JEPA-style comparison we crop the MI interval starting at 2s after resampling.
MI_WINDOW_START_S = 2.0
MI_WINDOW_START_SAMPLE = int(round(MI_WINDOW_START_S * SFREQ))
MI_WINDOW_STOP_SAMPLE = MI_WINDOW_START_SAMPLE + TARGET_WINDOW_SAMPLES

CV_FOLDS = 5
RANDOM_STATE = 2026
ASSERT_BALANCED_FOLDS = True

# CSP/FBCSP setup from your S-JEPA-style baseline family.
CSP_BAND = (8.0, 30.0)
FBCSP_BANDS = [
    (8, 12), (9, 13), (10, 14), (11, 15), (12, 16),
    (13, 17), (14, 18), (15, 19), (16, 20), (17, 21),
    (18, 22), (19, 23), (20, 24), (21, 25), (22, 26),
    (23, 27), (24, 28), (25, 29), (26, 30),
]
N_CSP_COMPONENTS = 4
PAPER_TARGETS = {'CSP_LDA': 55.57, 'FBCSP_SVM': 57.57}

# Source channel convention from Liu paper / observed source files:
# rawdata shape: trials x 33 channels x 4000 samples
# 0..29 = EEG-like channels, index 17 = CPz reference channel, 30..31 = EOG, 32 = marker.
# MOABB exposes 29 EEG channels, so we drop CPz reference, EOG, and marker.
EEG_CHANNEL_INDICES_29 = [i for i in range(30) if i != 17]
EEG_CHANNEL_NAMES_29 = [
    'Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'FCz', 'FC3', 'FC4',
    'FT7', 'FT8', 'Cz', 'C3', 'C4', 'T3', 'T4',
    # skip CPz original reference at source index 17
    'CP3', 'CP4', 'TP7', 'TP8', 'Pz', 'P3', 'P4', 'T5', 'T6', 'Oz', 'O1', 'O2'
]

print(f'Artifacts: {ARTIFACT_DIR}')
print(f'Target window: {TARGET_WINDOW_SAMPLES} samples = {TARGET_WINDOW_DURATION_S:.6f}s')
print(f'Crop: {MI_WINDOW_START_SAMPLE}:{MI_WINDOW_STOP_SAMPLE} at {SFREQ} Hz')
print(f'Balanced fold assertions: {ASSERT_BALANCED_FOLDS}')
print(f'FBCSP bands: {FBCSP_BANDS[0]} ... {FBCSP_BANDS[-1]} ({len(FBCSP_BANDS)} bands)')

In [ ]:
_log_handle = open(LOG_PATH, 'w', buffering=1)

def log(msg=''):
    text = str(msg)
    print(text)
    _log_handle.write(text + '\n')

log('Liu2024 source-mat S-JEPA-style preprocessing CSP/FBCSP run')
log(f'Artifacts: {ARTIFACT_DIR}')
log(f'Target window samples: {TARGET_WINDOW_SAMPLES}')
log(f'Effective target duration: {TARGET_WINDOW_DURATION_S:.6f}s')

## 2. Locate Liu2024 source `.mat` files

This should point to the extracted Figshare `sourcedata` folder, not MOABB cache files.

In [ ]:
def find_source_mat_files(root: Path):
    root = Path(root).expanduser()
    if not root.exists():
        return []

    # Prefer the actual Liu2024 subject files instead of any unrelated MATLAB files
    # that may exist in the project/code folders.
    subject_files = sorted(root.rglob('sub-*_task-motor-imagery_eeg.mat'))
    if subject_files:
        return subject_files

    # Fallback for slightly different Figshare/BIDS naming.
    subject_files = sorted([
        p for p in root.rglob('*.mat')
        if re.search(r'sub[-_]\d+', p.name, flags=re.IGNORECASE)
        and re.search(r'motor[-_]?imagery|motor', p.name, flags=re.IGNORECASE)
    ])
    if subject_files:
        return subject_files

    return sorted(root.rglob('*.mat'))


def _cwd_and_parents(max_depth=6):
    cwd = Path.cwd().resolve()
    roots = [cwd]
    for p in list(cwd.parents)[:max_depth]:
        roots.append(p)
    return roots

MAT_FILES = []
files = find_source_mat_files(SOURCE_EXTRACT_DIR)
if files:
    MAT_FILES = files

log(f'Source extract dir: {SOURCE_EXTRACT_DIR}')
log(f'Found {len(MAT_FILES)} .mat files')
log('First 5 files:')
for p in MAT_FILES[:5]:
    log(f'  {p}')


## 3. Robust source `.mat` loader

The Figshare files may store arrays under an `eeg` MATLAB struct, not top-level `rawdata` / `labels`. This loader recursively searches nested structs and verifies that it found a 3D data array plus a label vector.

In [ ]:
def subject_id_from_path(path: Path):
    s = str(path)
    m = re.search(r'sub[-_ ]?(\d{1,2})', s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r'\d+', Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f'Could not infer subject id from path: {path}')


def _is_mat_struct(x):
    return hasattr(x, '_fieldnames')


def _walk_mat_object(obj, prefix=''):
    # Yield (name, value) recursively from scipy-loaded MATLAB structs.
    # Liu2024 files may expose only a top-level `eeg` object.
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k.startswith('__'):
                continue
            name = f'{prefix}.{k}' if prefix else k
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f'{prefix}.{k}' if prefix else k
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                name = f'{prefix}{idx}'
                yield from _walk_mat_object(item, name)


def mat_structure_preview(path: Path, max_rows=200):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    rows = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            rows.append({'name': name, 'type': 'ndarray', 'shape': str(value.shape), 'dtype': str(value.dtype)})
        else:
            rows.append({'name': name, 'type': type(value).__name__, 'shape': '', 'dtype': ''})
    return pd.DataFrame(rows).head(max_rows)


def _normalize_rawdata_shape(rawdata, labels=None):
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f'rawdata must be 3D, got {arr.shape}')

    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []

    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]

    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)

    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)

    if arr.shape[1] < 29 or arr.shape[2] < 1000:
        raise ValueError(f'Could not normalize rawdata to trials x channels x samples, got {arr.shape}')
    return arr


def _score_raw_candidate(name, arr):
    lname = name.lower()
    score = 0
    if 'rawdata' in lname or 'raw' in lname or 'data' in lname:
        score += 10
    if arr.ndim == 3:
        score += 5
    if any(size in (39, 40) for size in arr.shape):
        score += 3
    if max(arr.shape) >= 3000:
        score += 2
    if 'eeg' in lname:
        score += 1
    return score


def _score_label_candidate(name, arr):
    lname = name.lower()
    flat = np.asarray(arr).ravel()
    unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
    score = 0
    if 'label' in lname or 'class' in lname or lname.split('.')[-1] in {'y', 'labels'}:
        score += 10
    if flat.size in (39, 40):
        score += 3
    if unique and unique.issubset({'0', '1', '2'}):
        score += 2
    return score


def load_subject_mat(path: Path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    arrays = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.dtype != object:
            arrays.append((name, np.asarray(value)))

    raw_candidates = []
    label_candidates = []
    for name, arr in arrays:
        if arr.ndim == 3:
            raw_candidates.append((_score_raw_candidate(name, arr), name, arr))
        elif arr.ndim in (1, 2):
            label_candidates.append((_score_label_candidate(name, arr), name, arr))

    if not raw_candidates or not label_candidates:
        preview = mat_structure_preview(path)
        preview_path = ARTIFACT_DIR / f'mat_structure_failure_{Path(path).stem}.csv'
        preview.to_csv(preview_path, index=False)
        log(f'MAT structure preview for failure saved to: {preview_path}')
        log(preview.to_string(index=False))
        raise KeyError(f'Could not locate 3D raw data and labels in {path}')

    raw_candidates = sorted(raw_candidates, key=lambda x: x[0], reverse=True)
    label_candidates = sorted(label_candidates, key=lambda x: x[0], reverse=True)
    raw_name, raw_arr = raw_candidates[0][1], np.asarray(raw_candidates[0][2])
    label_name, label_arr = label_candidates[0][1], np.asarray(label_candidates[0][2]).astype(int).ravel()

    rawdata = _normalize_rawdata_shape(raw_arr, labels=label_arr)
    labels = np.asarray(label_arr).astype(int).ravel()

    if labels.size != rawdata.shape[0]:
        raise ValueError(f'Cannot align labels {labels.shape} with rawdata {rawdata.shape} in {path}')

    return rawdata.astype(np.float64), labels.astype(int), raw_name, label_name


preview = mat_structure_preview(MAT_FILES[0])
preview_path = ARTIFACT_DIR / 'mat_structure_subject_01.csv'
preview.to_csv(preview_path, index=False)
log(f'Wrote MAT structure preview: {preview_path}')
display(preview.head(20))


## 4. Load and verify all subjects

Expected: 50 subjects, 40 trials per subject, balanced 20/20 labels.

In [ ]:
subjects = []
for p in MAT_FILES:
    sid = subject_id_from_path(p)
    X_raw, y_raw, raw_field, label_field = load_subject_mat(p)
    subjects.append({
        'subject_id': sid,
        'path': str(p),
        'rawdata_shape': tuple(X_raw.shape),
        'labels_shape': tuple(y_raw.shape),
        'label_counts_raw': np.bincount(y_raw.astype(int), minlength=3).tolist(),
        'raw_field': raw_field,
        'label_field': label_field,
    })

subjects_df = pd.DataFrame(subjects).sort_values('subject_id')
subjects_df.to_csv(ARTIFACT_DIR / 'source_mat_summary.csv', index=False)
log(f'Wrote source summary: {ARTIFACT_DIR / "source_mat_summary.csv"}')
display(subjects_df.head())
log(subjects_df[['subject_id', 'rawdata_shape', 'labels_shape', 'label_counts_raw', 'raw_field', 'label_field']].head().to_string(index=False))

## 5. MOABB-style S-JEPA preprocessing from Liu source trials

This cell converts each subject into fixed windows as arrays:

```text
source .mat trials: 40 × 33 × 4000 at 500 Hz
select EEG only:    40 × 29 × 4000
resample/filter/ref:40 × 29 × 1024 at 128 Hz
crop MI window:     40 × 29 × 537
```

The raw-level preprocessing order now mirrors the MOABB S-JEPA notebook:

```text
pick/select EEG → average reference → resample 128 Hz → filter 0.5–40 Hz → scale to µV
```

Because Liu source `.mat` files are already epoched, the notebook concatenates the 40 trials into one subject-level `RawArray`, applies the MOABB-style raw preprocessing, then reshapes back into trials before cropping the fixed S-JEPA-compatible MI window.


In [ ]:
def make_liu_info(sfreq):
    info = mne.create_info(
        ch_names=EEG_CHANNEL_NAMES_29,
        sfreq=float(sfreq),
        ch_types=['eeg'] * len(EEG_CHANNEL_NAMES_29),
    )
    try:
        montage = mne.channels.make_standard_montage('standard_1020')
        info.set_montage(montage, match_case=False, on_missing='ignore')
    except Exception:
        pass
    return info


def source_microvolts_to_mne_volts(data):
    """Hard-coded Liu source convention.

    The Liu source `.mat` values are treated as microvolts. MNE RawArray expects
    volts, so convert before applying the same raw-level preprocessing order used
    by the MOABB S-JEPA notebook.
    """
    return np.asarray(data, dtype=np.float64) * 1e-6


def scale_volts_to_microvolts(data):
    """Same final scaling step used by the MOABB S-JEPA preprocessing path."""
    return np.asarray(data, dtype=np.float64) * 1e6


def labels_to_zero_based(labels, subject_id):
    labels = np.asarray(labels).astype(int).ravel()
    unique_labels = set(np.unique(labels).tolist())
    if unique_labels.issubset({1, 2}):
        y = labels - 1
    elif unique_labels.issubset({0, 1}):
        y = labels
    else:
        raise ValueError(f'Unexpected raw labels for subject {subject_id}: {np.unique(labels)}')

    if not set(np.unique(y)).issubset({0, 1}):
        raise ValueError(f'Unexpected labels after conversion for subject {subject_id}: {np.unique(y)}')
    return y.astype(int)


def preprocess_subject_sjepa_style(rawdata, labels, subject_id):
    """Apply MOABB-style S-JEPA preprocessing to one Liu2024 source subject.

    MOABB reference path:
        pick EEG -> average reference -> resample -> bandpass -> scale to microvolts

    Liu source MAT adaptation:
        select 29 EEG channels/drop CPz+EOG+marker
        -> convert source microvolts to MNE volts
        -> average reference
        -> resample
        -> bandpass
        -> scale to microvolts
        -> fixed S-JEPA-compatible crop
    """
    if rawdata.ndim != 3:
        raise ValueError(f'Subject {subject_id}: expected 3D rawdata, got {rawdata.shape}')
    if rawdata.shape[1] < 33:
        raise ValueError(f'Subject {subject_id}: expected at least 33 channels, got {rawdata.shape}')
    if rawdata.shape[2] != 4000:
        log(f'WARNING subject {subject_id}: expected 4000 samples, got {rawdata.shape[2]}')

    n_trials = rawdata.shape[0]

    # Equivalent to MOABB/Braindecode pick_types(eeg=True, meg=False, stim=False),
    # adapted to the Liu source MAT layout. This always drops EOG, marker, and CPz reference.
    X_eeg = rawdata[:, EEG_CHANNEL_INDICES_29, :].astype(np.float64)

    # MNE RawArray expects EEG data in volts. The final downstream input is scaled
    # back to microvolts, matching the MOABB S-JEPA notebook's final scaling step.
    X_eeg_volts = source_microvolts_to_mne_volts(X_eeg)

    # Concatenate trials into one continuous RawArray so preprocessing is applied
    # at the same raw-object level as the MOABB pipeline.
    continuous = X_eeg_volts.transpose(1, 0, 2).reshape(len(EEG_CHANNEL_INDICES_29), -1)
    raw = mne.io.RawArray(continuous, make_liu_info(SOURCE_SFREQ), verbose=False)

    # Keep this order aligned with moabb_mi_sjepa.ipynb:
    #   Preprocessor("set_eeg_reference", ref_channels="average")
    #   Preprocessor("resample", sfreq=CONFIG["sfreq"])
    #   Preprocessor("filter", l_freq=CONFIG["bandpass_low"], h_freq=CONFIG["bandpass_high"])
    #   Preprocessor(scale_volts_to_microvolts)
    raw.set_eeg_reference('average', projection=False, verbose=False)
    raw.resample(SFREQ, verbose=False)
    raw.filter(BANDPASS_LOW, BANDPASS_HIGH, verbose=False)

    data = scale_volts_to_microvolts(raw.get_data())

    expected_samples_per_trial = int(round(rawdata.shape[2] * SFREQ / SOURCE_SFREQ))
    total_expected = n_trials * expected_samples_per_trial

    if data.shape[1] != total_expected:
        n_full = data.shape[1] // n_trials
        log(
            f'WARNING subject {subject_id}: resampled samples {data.shape[1]} != expected {total_expected}; '
            f'using {n_full} samples/trial.'
        )
        expected_samples_per_trial = n_full
        data = data[:, :n_trials * expected_samples_per_trial]

    X_rs = data.reshape(len(EEG_CHANNEL_INDICES_29), n_trials, expected_samples_per_trial).transpose(1, 0, 2)

    if MI_WINDOW_STOP_SAMPLE > X_rs.shape[-1]:
        raise ValueError(
            f'MI crop {MI_WINDOW_START_SAMPLE}:{MI_WINDOW_STOP_SAMPLE} exceeds resampled trial length {X_rs.shape[-1]}'
        )

    X_win = X_rs[:, :, MI_WINDOW_START_SAMPLE:MI_WINDOW_STOP_SAMPLE]
    y = labels_to_zero_based(labels, subject_id)

    return X_win.astype(np.float32), y.astype(int), X_rs.shape[-1]


Xs, ys, subjects_arr = [], [], []
window_summary_rows = []

for item in subjects_df.to_dict('records'):
    sid = int(item['subject_id'])
    X_raw, y_raw, raw_field, label_field = load_subject_mat(Path(item['path']))
    X_win, y, samples_per_trial = preprocess_subject_sjepa_style(X_raw, y_raw, sid)

    Xs.append(X_win)
    ys.append(y)
    subjects_arr.extend([sid] * len(y))

    window_summary_rows.append({
        'subject_id': sid,
        'n_windows': int(len(y)),
        'class_counts': np.bincount(y, minlength=2).tolist(),
        'preprocessed_shape': tuple(X_win.shape),
        'resampled_samples_per_trial': int(samples_per_trial),
        'crop_start_sample': MI_WINDOW_START_SAMPLE,
        'crop_stop_sample': MI_WINDOW_STOP_SAMPLE,
        'target_window_samples': TARGET_WINDOW_SAMPLES,
        'effective_window_duration_s': TARGET_WINDOW_DURATION_S,
    })

X_all = np.concatenate(Xs, axis=0)
y_all = np.concatenate(ys, axis=0)
subjects_arr = np.asarray(subjects_arr)

window_summary_df = pd.DataFrame(window_summary_rows).sort_values('subject_id')
window_summary_df.to_csv(ARTIFACT_DIR / 'window_counts_by_subject.csv', index=False)

log(f'X_all shape: {X_all.shape}')
log(f'y_all counts: {np.bincount(y_all, minlength=2).tolist()}')
log(f'subjects_arr shape: {subjects_arr.shape}')
log(f'Wrote window summary: {ARTIFACT_DIR / "window_counts_by_subject.csv"}')
display(window_summary_df.head())


## 6. CSP/FBCSP helpers

Only one CSP setup and one FBCSP setup are used here.

In [ ]:
def bandpass_zero_phase(X, sfreq, l_freq, h_freq, order=5):
    sos = butter(order, [l_freq, h_freq], btype='bandpass', fs=sfreq, output='sos')
    return sosfiltfilt(sos, X, axis=-1)


def make_csp_lda(n_components=N_CSP_COMPONENTS):
    return Pipeline([
        ('csp', CSP(n_components=n_components, reg='ledoit_wolf', log=True, norm_trace=False)),
        ('lda', LinearDiscriminantAnalysis()),
    ])


def make_fbcsp_features(X_train, y_train, X_test):
    feats_train, feats_test = [], []
    for band in FBCSP_BANDS:
        Xtr = bandpass_zero_phase(X_train, SFREQ, band[0], band[1])
        Xte = bandpass_zero_phase(X_test, SFREQ, band[0], band[1])
        csp = CSP(n_components=N_CSP_COMPONENTS, reg='ledoit_wolf', log=True, norm_trace=False)
        feats_train.append(csp.fit_transform(Xtr, y_train))
        feats_test.append(csp.transform(Xte))
    return np.concatenate(feats_train, axis=1), np.concatenate(feats_test, axis=1)


def make_fbcsp_svm():
    return Pipeline([
        ('scale', StandardScaler()),
        ('svm', SVC(kernel='linear', C=1.0)),
    ])


def fold_metrics(y_test, pred, scores=None):
    row = {
        'accuracy': float(accuracy_score(y_test, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test, pred)),
        'confusion_matrix': confusion_matrix(y_test, pred, labels=[0, 1]).tolist(),
        'prediction_histogram': np.bincount(pred, minlength=2).tolist(),
    }
    if scores is not None and len(np.unique(y_test)) == 2:
        try:
            row['roc_auc'] = float(roc_auc_score(y_test, scores))
        except Exception:
            row['roc_auc'] = None
    else:
        row['roc_auc'] = None
    return row


## 7. Within-subject 5-fold CSP/FBCSP evaluation

This follows the S-JEPA-style downstream comparison logic: within-subject stratified 5-fold CV.

In [ ]:
rows = []

def _assert_expected_fold_balance(y_train, y_test, subject_id, fold_id):
    if not ASSERT_BALANCED_FOLDS:
        return
    train_counts = np.bincount(y_train, minlength=2)
    test_counts = np.bincount(y_test, minlength=2)
    expected_train = np.array([16, 16])
    expected_test = np.array([4, 4])
    assert np.array_equal(train_counts, expected_train), (
        f'Subject {subject_id} fold {fold_id}: train counts {train_counts.tolist()} != {expected_train.tolist()}'
    )
    assert np.array_equal(test_counts, expected_test), (
        f'Subject {subject_id} fold {fold_id}: test counts {test_counts.tolist()} != {expected_test.tolist()}'
    )

for subj in sorted(pd.unique(subjects_arr), key=lambda x: int(x)):
    idx = np.where(subjects_arr == subj)[0]
    X = X_all[idx]
    y = y_all[idx]
    counts = np.bincount(y, minlength=2)
    log(f'Subject {subj}: X={X.shape}, class_counts={counts.tolist()}')

    if counts.min() < CV_FOLDS:
        log(f'  SKIP subject {subj}: not enough trials per class for {CV_FOLDS}-fold CV')
        continue

    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    for fold, (tr, te) in enumerate(cv.split(X, y), start=1):
        X_train_raw, X_test_raw = X[tr], X[te]
        y_train, y_test = y[tr], y[te]
        _assert_expected_fold_balance(y_train, y_test, subj, fold)

        # CSP + LDA: 8-30 Hz over the S-JEPA-preprocessed/cropped windows.
        X_train = bandpass_zero_phase(X_train_raw, SFREQ, CSP_BAND[0], CSP_BAND[1])
        X_test = bandpass_zero_phase(X_test_raw, SFREQ, CSP_BAND[0], CSP_BAND[1])
        clf = make_csp_lda()
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        scores = clf.decision_function(X_test) if hasattr(clf, 'decision_function') else None
        row = {
            'subject_id': str(subj),
            'fold_id': fold,
            'model_name': 'CSP_LDA',
            'n_train': int(len(tr)),
            'n_test': int(len(te)),
            'train_class_counts': np.bincount(y_train, minlength=2).tolist(),
            'test_class_counts': np.bincount(y_test, minlength=2).tolist(),
            'feature_band': list(CSP_BAND),
            'n_csp_components': N_CSP_COMPONENTS,
        }
        row.update(fold_metrics(y_test, pred, scores))
        rows.append(row)

        # FBCSP + SVM: Liu2024-style naming with overlapping 8-30 Hz filter bank,
        # evaluated on the same S-JEPA-preprocessed/cropped windows.
        Ftr, Fte = make_fbcsp_features(X_train_raw, y_train, X_test_raw)
        svm = make_fbcsp_svm()
        svm.fit(Ftr, y_train)
        pred = svm.predict(Fte)
        scores = svm.decision_function(Fte) if hasattr(svm, 'decision_function') else None
        row = {
            'subject_id': str(subj),
            'fold_id': fold,
            'model_name': 'FBCSP_SVM',
            'n_train': int(len(tr)),
            'n_test': int(len(te)),
            'train_class_counts': np.bincount(y_train, minlength=2).tolist(),
            'test_class_counts': np.bincount(y_test, minlength=2).tolist(),
            'n_filter_bands': len(FBCSP_BANDS),
            'n_csp_components_per_band': N_CSP_COMPONENTS,
        }
        row.update(fold_metrics(y_test, pred, scores))
        rows.append(row)

results_df = pd.DataFrame(rows)
results_path = ARTIFACT_DIR / 'cv_results.csv'
results_df.to_csv(results_path, index=False)
log(f'Wrote fold results: {results_path}')
display(results_df.head())


## 8. Aggregate metrics and diagnostics

In [ ]:
import matplotlib.pyplot as plt

global_metrics = (
    results_df
    .groupby('model_name')
    .agg(
        mean_accuracy=('accuracy', 'mean'),
        std_accuracy=('accuracy', 'std'),
        mean_balanced_accuracy=('balanced_accuracy', 'mean'),
        std_balanced_accuracy=('balanced_accuracy', 'std'),
        mean_roc_auc=('roc_auc', 'mean'),
        std_roc_auc=('roc_auc', 'std'),
        n_folds_total=('fold_id', 'count'),
        n_subjects=('subject_id', lambda s: int(pd.Series(s).nunique())),
    )
    .reset_index()
)

global_metrics['paper_target_accuracy'] = global_metrics['model_name'].map(PAPER_TARGETS).astype(float)
global_metrics['mean_accuracy_percent'] = global_metrics['mean_accuracy'] * 100.0
global_metrics['delta_vs_paper_percent'] = global_metrics['mean_accuracy_percent'] - global_metrics['paper_target_accuracy']

global_metrics_path = ARTIFACT_DIR / 'global_metrics.csv'
global_metrics.to_csv(global_metrics_path, index=False)

# Alias using the naming convention from the author-match notebook.
global_method_comparison_path = ARTIFACT_DIR / 'global_method_comparison.csv'
global_metrics.rename(columns={'model_name': 'method_name'}).to_csv(global_method_comparison_path, index=False)

log('Global metrics:')
log(global_metrics.to_string(index=False))
display(global_metrics)

subject_metrics = (
    results_df
    .groupby(['model_name', 'subject_id'])
    .agg(
        mean_accuracy=('accuracy', 'mean'),
        std_accuracy=('accuracy', 'std'),
        mean_balanced_accuracy=('balanced_accuracy', 'mean'),
        mean_roc_auc=('roc_auc', 'mean'),
        n_folds=('fold_id', 'count'),
    )
    .reset_index()
)
subject_metrics['mean_accuracy_percent'] = subject_metrics['mean_accuracy'] * 100.0
subject_metrics_path = ARTIFACT_DIR / 'subject_metrics.csv'
subject_metrics.to_csv(subject_metrics_path, index=False)

# Alias using the naming convention from the author-match notebook.
subject_level_summary_path = ARTIFACT_DIR / 'subject_level_summary.csv'
subject_metrics.rename(columns={'model_name': 'method_name'}).to_csv(subject_level_summary_path, index=False)

collapse_rows = []
for model_name, sub in results_df.groupby('model_name'):
    n_folds = len(sub)
    n_collapsed = 0
    counts = {0: 0, 1: 0}
    for hist in sub['prediction_histogram']:
        h = hist if isinstance(hist, list) else json.loads(hist)
        nonzero = [i for i, v in enumerate(h) if v > 0]
        if len(nonzero) == 1:
            n_collapsed += 1
            counts[nonzero[0]] += 1
    collapse_rows.append({
        'model_name': model_name,
        'n_folds': int(n_folds),
        'n_collapsed_single_class': int(n_collapsed),
        'collapsed_fraction': float(n_collapsed / n_folds) if n_folds else None,
        'single_class_prediction_counts': counts,
    })
collapse_df = pd.DataFrame(collapse_rows)
collapse_path = ARTIFACT_DIR / 'collapse_diagnostics.csv'
collapse_df.to_csv(collapse_path, index=False)
log('Collapse diagnostics:')
log(collapse_df.to_string(index=False))
display(collapse_df)


def _parse_confusion_matrix(value):
    if isinstance(value, list):
        return np.asarray(value, dtype=int)
    if isinstance(value, np.ndarray):
        return value.astype(int)
    return np.asarray(json.loads(value), dtype=int)


def aggregate_confusion_matrix(results, model_name, author_order=True):
    mats = [_parse_confusion_matrix(v) for v in results.loc[results['model_name'] == model_name, 'confusion_matrix']]
    cm = np.sum(mats, axis=0).astype(int)
    if author_order:
        # Stored order is [Left, Right] = [0, 1]. Paper visual order is [Right, Left].
        cm = cm[np.ix_([1, 0], [1, 0])]
    return cm


def plot_sjepa_cv_confusion_matrices(results):
    methods = [('CSP_LDA', 'CSP+LDA'), ('FBCSP_SVM', 'FBCSP+SVM')]
    fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8), constrained_layout=True)
    rows = []
    for ax, (model_name, title), panel in zip(axes, methods, ['a', 'b']):
        cm = aggregate_confusion_matrix(results, model_name, author_order=True)
        im = ax.imshow(cm)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Predict class', fontweight='bold')
        ax.set_ylabel('Actual class', fontweight='bold')
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Right Hand', 'Left Hand'])
        ax.set_yticks([0, 1])
        ax.set_yticklabels(['Right Hand', 'Left Hand'], rotation=90, va='center')
        ax.text(-0.22, 1.04, panel, transform=ax.transAxes, fontsize=12, fontweight='bold')
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(int(cm[i, j])), ha='center', va='center', fontsize=13)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        rows.append({
            'model_name': model_name,
            'author_style_confusion_matrix_right_left_order': cm.tolist(),
            'n_predictions': int(cm.sum()),
            'accuracy_from_matrix': float((cm[0, 0] + cm[1, 1]) / cm.sum() * 100.0),
        })
    out_path = ARTIFACT_DIR / 'sjepa_cv_confusion_matrices_csp_fbcsp.png'
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()
    metrics_path = ARTIFACT_DIR / 'sjepa_cv_confusion_metrics.csv'
    pd.DataFrame(rows).to_csv(metrics_path, index=False)
    log(f'Wrote S-JEPA CV confusion matrix plot: {out_path}')
    log(f'Wrote S-JEPA CV confusion metrics: {metrics_path}')
    return out_path, metrics_path


def plot_subject_accuracy_paper_style(subject_metrics_df):
    df = subject_metrics_df[subject_metrics_df['model_name'].isin(['CSP_LDA', 'FBCSP_SVM'])].copy()
    df['subject_id_int'] = df['subject_id'].astype(int)
    wide = (
        df.pivot(index='subject_id_int', columns='model_name', values='mean_accuracy_percent')
        .sort_index()
    )

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharey=True)
    for ax, subject_ids, panel_label in [
        (axes[0], list(range(1, 26)), 'a'),
        (axes[1], list(range(26, 51)), 'b'),
    ]:
        available_ids = [sid for sid in subject_ids if sid in wide.index]
        data = wide.loc[available_ids]
        x = np.arange(len(data))
        width = 0.38
        ax.bar(x - width / 2, data['CSP_LDA'], width, label='CSP+LDA')
        ax.bar(x + width / 2, data['FBCSP_SVM'], width, label='FBCSP+SVM')

        ax.axhline(PAPER_TARGETS['CSP_LDA'], linestyle='--', linewidth=1, label='Paper CSP+LDA 55.57%' if panel_label == 'a' else None)
        ax.axhline(PAPER_TARGETS['FBCSP_SVM'], linestyle=':', linewidth=1.5, label='Paper FBCSP+SVM 57.57%' if panel_label == 'a' else None)
        ax.set_ylim(0, 100)
        ax.set_ylabel('Classification Accuracy')
        ax.set_xticks(x)
        ax.set_xticklabels(available_ids)
        ax.grid(axis='y', alpha=0.25)
        ax.text(-0.07, 1.03, panel_label, transform=ax.transAxes, fontweight='bold', fontsize=12)

    axes[0].legend(loc='upper right', ncol=4, frameon=True)
    axes[1].set_xlabel('Subject')
    fig.suptitle('Liu2024 CSP/FBCSP subject-level accuracy', fontsize=16)
    fig.tight_layout()

    out_path = ARTIFACT_DIR / 'liu2024_csp_fbcsp_subject_bars_sjepa_style.png'
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()
    log(f'Wrote subject-level S-JEPA-style plot: {out_path}')
    return out_path


confusion_plot_path, confusion_metrics_path = plot_sjepa_cv_confusion_matrices(results_df)
subject_plot_path = plot_subject_accuracy_paper_style(subject_metrics)


## 9. Additional Performance Visualizations

This section adds report-ready performance visuals: global mean accuracy with uncertainty, subject-level accuracy with global mean lines, and subject-mean distributions. These plots complement the confusion matrices and make it easier to judge whether the baseline is globally above chance or driven by only a few subjects.


In [ ]:
import matplotlib.pyplot as plt

def _model_label(model_name):
    return {
        "CSP_LDA": "CSP+LDA",
        "FBCSP_SVM": "FBCSP+SVM",
    }.get(model_name, str(model_name))


def plot_global_accuracy_summary(global_metrics_df):
    df = global_metrics_df.copy()
    df["label"] = df["model_name"].map(_model_label)
    x = np.arange(len(df))
    means = df["mean_accuracy_percent"].to_numpy(dtype=float)
    errors = (df["std_accuracy"].to_numpy(dtype=float) * 100.0)

    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    ax.bar(x, means, yerr=errors, capsize=5, alpha=0.78, label="Notebook mean ± fold SD")

    for pos, (_, row) in enumerate(df.iterrows()):
        target = row.get("paper_target_accuracy")
        if pd.notna(target):
            ax.scatter(pos, float(target), marker="D", s=90, label="Paper Table 4 target" if pos == 0 else None)
        ax.text(
            pos,
            means[pos] + errors[pos] + 2.0,
            f"{means[pos]:.2f}%\nΔ {row['delta_vs_paper_percent']:+.2f}",
            ha="center",
            va="bottom",
            fontsize=10,
        )

    ax.axhline(50.0, linestyle="--", linewidth=1.2, label="Chance level 50%")
    ax.set_xticks(x)
    ax.set_xticklabels(df["label"].tolist())
    ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.25)
    ax.set_title("Liu2024 S-JEPA-windowed CSP/FBCSP global accuracy\nmean ± SD across folds")
    ax.legend(loc="upper right", frameon=True)
    fig.tight_layout()

    out_path = ARTIFACT_DIR / "csp_fbcsp_global_accuracy_summary.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    log(f"Wrote global accuracy summary plot: {out_path}")
    return out_path


def plot_subject_accuracy_with_global_means(subject_metrics_df, global_metrics_df):
    df = subject_metrics_df[subject_metrics_df["model_name"].isin(["CSP_LDA", "FBCSP_SVM"])].copy()
    df["subject_id_int"] = df["subject_id"].astype(int)
    wide = df.pivot(index="subject_id_int", columns="model_name", values="mean_accuracy_percent").sort_index()
    global_mean = dict(zip(global_metrics_df["model_name"], global_metrics_df["mean_accuracy_percent"]))

    fig, axes = plt.subplots(2, 1, figsize=(16, 7.8), sharey=True)
    for ax, subject_ids, panel_label in [
        (axes[0], list(range(1, 26)), "a"),
        (axes[1], list(range(26, 51)), "b"),
    ]:
        available_ids = [sid for sid in subject_ids if sid in wide.index]
        data = wide.loc[available_ids]
        x = np.arange(len(data))
        width = 0.38

        ax.bar(x - width / 2, data["CSP_LDA"], width, label="CSP+LDA")
        ax.bar(x + width / 2, data["FBCSP_SVM"], width, label="FBCSP+SVM")

        ax.axhline(50.0, linestyle="--", linewidth=1.1, label="Chance level 50%" if panel_label == "a" else None)
        ax.axhline(global_mean.get("CSP_LDA", np.nan), linestyle="-.", linewidth=1.4, label=f"CSP+LDA mean {global_mean.get('CSP_LDA', np.nan):.2f}%" if panel_label == "a" else None)
        ax.axhline(global_mean.get("FBCSP_SVM", np.nan), linestyle=":", linewidth=1.7, label=f"FBCSP+SVM mean {global_mean.get('FBCSP_SVM', np.nan):.2f}%" if panel_label == "a" else None)
        ax.axhline(PAPER_TARGETS["CSP_LDA"], linestyle=(0, (5, 3)), linewidth=1.0, label="Paper CSP+LDA 55.57%" if panel_label == "a" else None)
        ax.axhline(PAPER_TARGETS["FBCSP_SVM"], linestyle=(0, (1, 2)), linewidth=1.3, label="Paper FBCSP+SVM 57.57%" if panel_label == "a" else None)

        ax.set_ylim(0, 100)
        ax.set_ylabel("Classification Accuracy (%)")
        ax.set_xticks(x)
        ax.set_xticklabels(available_ids)
        ax.grid(axis="y", alpha=0.25)
        ax.text(-0.055, 1.03, panel_label, transform=ax.transAxes, fontweight="bold", fontsize=13)

    axes[0].legend(loc="upper right", ncol=3, frameon=True, fontsize=9)
    axes[1].set_xlabel("Subject")
    fig.suptitle("Liu2024 S-JEPA-windowed CSP/FBCSP subject-level accuracy with means", fontsize=16)
    fig.tight_layout()

    out_path = ARTIFACT_DIR / "csp_fbcsp_subject_accuracy_with_means.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    log(f"Wrote subject accuracy with means plot: {out_path}")
    return out_path


def plot_subject_accuracy_distribution(subject_metrics_df, global_metrics_df):
    df = subject_metrics_df[subject_metrics_df["model_name"].isin(["CSP_LDA", "FBCSP_SVM"])].copy()
    methods = ["CSP_LDA", "FBCSP_SVM"]
    data = [
        df.loc[df["model_name"] == method, "mean_accuracy_percent"].dropna().to_numpy(dtype=float)
        for method in methods
    ]

    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    ax.boxplot(data, labels=[_model_label(m) for m in methods], showmeans=True)

    rng = np.random.default_rng(123)
    for pos, values in enumerate(data, start=1):
        jitter = rng.uniform(-0.06, 0.06, size=len(values))
        ax.scatter(np.full(len(values), pos) + jitter, values, s=22, alpha=0.65)

    for pos, method in enumerate(methods, start=1):
        paper_target = PAPER_TARGETS.get(method)
        model_global = float(global_metrics_df.loc[global_metrics_df["model_name"] == method, "mean_accuracy_percent"].iloc[0])
        if paper_target is not None:
            ax.scatter(pos, paper_target, marker="D", s=90, label="Paper Table 4 target" if pos == 1 else None)
        ax.scatter(pos, model_global, marker="^", s=90, label="Notebook global mean" if pos == 1 else None)

    ax.axhline(50.0, linestyle="--", linewidth=1.2, label="Chance level 50%")
    ax.set_ylabel("Subject mean accuracy (%)")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.25)
    ax.set_title("Distribution of Liu2024 subject-level mean accuracies")
    ax.legend(loc="upper right", frameon=True)
    fig.tight_layout()

    out_path = ARTIFACT_DIR / "csp_fbcsp_subject_accuracy_distribution.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    log(f"Wrote subject accuracy distribution plot: {out_path}")
    return out_path


global_accuracy_plot_path = plot_global_accuracy_summary(global_metrics)
subject_accuracy_with_means_plot_path = plot_subject_accuracy_with_global_means(subject_metrics, global_metrics)
subject_distribution_plot_path = plot_subject_accuracy_distribution(subject_metrics, global_metrics)


## 10. Save run metadata

In [ ]:
metadata = {
    'notebook': 'liu2024_source_mat_sjepa_preprocess_csp_fbcsp',
    'source': 'original Figshare sourcedata .mat files',
    'source_extract_dir': str(SOURCE_EXTRACT_DIR),
    'n_subjects': int(subjects_df['subject_id'].nunique()),
    'source_sfreq': SOURCE_SFREQ,
    'target_sfreq': SFREQ,
    'preprocessing_order': [
        'select 29 EEG channels; drop CPz source reference, EOG, and marker',
        'treat Liu source values as microvolts and convert to MNE volts',
        'concatenate source trials per subject into MNE RawArray',
        'average reference',
        'resample to 128 Hz',
        'bandpass 0.5-40 Hz',
        'scale volts to microvolts',
        'reshape back to trials',
        'crop fixed 537-sample S-JEPA window starting at 2.0s',
    ],
    'source_values_assumed': 'microvolts',
    'final_downstream_units': 'microvolts',
    'target_window_samples': TARGET_WINDOW_SAMPLES,
    'effective_window_duration_s': TARGET_WINDOW_DURATION_S,
    'mi_window_start_s': MI_WINDOW_START_S,
    'mi_window_start_sample': MI_WINDOW_START_SAMPLE,
    'mi_window_stop_sample': MI_WINDOW_STOP_SAMPLE,
    'channel_names': EEG_CHANNEL_NAMES_29,
    'channel_convention': 'Liu2024 paper / EEGLAB names; CPz source reference dropped',
    'cv': {
        'type': 'StratifiedKFold',
        'n_splits': CV_FOLDS,
        'shuffle': True,
        'random_state': RANDOM_STATE,
        'assert_balanced_folds': ASSERT_BALANCED_FOLDS,
        'expected_train_class_counts_per_fold': [16, 16],
        'expected_test_class_counts_per_fold': [4, 4],
    },
    'paper_targets': PAPER_TARGETS,
    'csp': {
        'model': 'CSP_LDA', 'feature_band': list(CSP_BAND), 'n_components': N_CSP_COMPONENTS,
        'reg': 'ledoit_wolf', 'log': True, 'norm_trace': False,
    },
    'fbcsp': {
        'model': 'FBCSP_SVM', 'bands': [list(b) for b in FBCSP_BANDS],
        'n_components_per_band': N_CSP_COMPONENTS, 'reg': 'ledoit_wolf', 'log': True, 'norm_trace': False,
        'svm': {'kernel': 'linear', 'C': 1.0, 'standardize_features': True},
    },
    'outputs': {
        'cv_results': str(results_path),
        'global_metrics': str(global_metrics_path),
        'global_method_comparison': str(global_method_comparison_path),
        'subject_metrics': str(subject_metrics_path),
        'subject_level_summary': str(subject_level_summary_path),
        'collapse_diagnostics': str(collapse_path),
        'confusion_plot': str(confusion_plot_path),
        'confusion_metrics': str(confusion_metrics_path),
        'subject_plot': str(subject_plot_path),
        'global_accuracy_plot': str(global_accuracy_plot_path) if 'global_accuracy_plot_path' in globals() else None,
        'subject_accuracy_with_means_plot': str(subject_accuracy_with_means_plot_path) if 'subject_accuracy_with_means_plot_path' in globals() else None,
        'subject_accuracy_distribution_plot': str(subject_distribution_plot_path) if 'subject_distribution_plot_path' in globals() else None,
    },
}

metadata_path = ARTIFACT_DIR / 'run_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
log(f'Wrote metadata: {metadata_path}')
log('Done.')
_log_handle.close()
